In [2]:
import pandas as pd
import random

# Templates combinatórios estruturados
templates = {
    'investimentos': {
        's': ['', 'Olá', 'Bom dia', 'Por favor', 'Gostaria de saber'],
        'a': ['como aplicar em', 'quero investir em', 'qual a rentabilidade do', 'como funciona o', 'desejo aplicar no'],
        'o': ['tesouro direto', 'cdb de liquidez diaria', 'fundo de investimento', 'lci e lca', 'mercado de acoes']
    },
    'consultas': {
        's': ['', 'Oi', 'Por gentileza', 'Pode me mostrar', 'Preciso ver'],
        'a': ['quero consultar', 'onde vejo', 'qual e o', 'mostre o', 'solicito o'],
        'o': ['meu saldo atual', 'extrato da minha conta', 'comprovante de transferencia', 'saldo disponivel', 'historico de transacoes']
    },
    'pagamentos': {
        's': ['', 'Olá bot', 'Bom dia', 'Urgente', 'Por favor'],
        'a': ['quero pagar', 'como faço para quitar', 'preciso agendar o pagamento do', 'como envio um', 'desejo pagar o'],
        'o': ['boleto de luz', 'codigo de barras', 'pix para chave email', 'cartao de credito', 'imposto veicular']
    },
    'financiamentos': {
        's': ['', 'Olá', 'Gostaria de simular', 'Por gentileza', 'Preciso de ajuda com'],
        'a': ['como contratar', 'quero simular um', 'quais as taxas do', 'como funciona a quitacao do', 'solicito proposta de'],
        'o': ['financiamento imobiliario', 'credito auto', 'financiamento de veiculo', 'credito com garantia', 'parcelamento da casa propria']
    }
}

amostras = []
random.seed(42) # Garantir reprodutibilidade exata das 100 frases

# Gerando exatamente 25 frases por intenção (Total = 100)
for intencao, comp in templates.items():
    for _ in range(25):
        s = random.choice(comp['s'])
        a = random.choice(comp['a'])
        o = random.choice(comp['o'])
        frase = f"{s} {a} {o}".strip().capitalize()
        amostras.append({'texto': frase, 'intencao': intencao})

# Exportação para arquivo CSV
df_banco = pd.DataFrame(amostras)
df_banco.to_csv('dataset_banco_100.csv', index=False, encoding='utf-8')

print(" Arquivo 'dataset_banco_100.csv' gerado com sucesso!")




 Arquivo 'dataset_banco_100.csv' gerado com sucesso!


In [3]:
import numpy as np
import pandas as pd
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. CRIAR O DATASET CORRETO DE MÓVEIS
# ============================================================

templates = {
    'vendas': {
        's': ['', 'Olá', 'Bom dia', 'Gostaria de saber', 'Por favor'],
        'a': [
            'quero comprar',
            'qual o preco do',
            'tem cupom para',
            'como faco para adquirir',
            'desejo orcamento de'
        ],
        'o': [
            'sofa retratil 3 lugares',
            'conjunto de mesa de jantar',
            'guarda roupa casal',
            'painel para tv',
            'colchao queen size'
        ]
    },

    'suporte': {
        's': ['', 'Oi', 'Preciso de ajuda', 'Por gentileza', 'Socorro'],
        'a': [
            'como montar o',
            'onde baixo o manual do',
            'estou com duvida no',
            'veio faltando parafuso no',
            'preciso de assistencia para'
        ],
        'o': [
            'armario de cozinha',
            'rack da sala',
            'berco do bebe',
            'esquema de montagem',
            'manual da estante'
        ]
    },

    'trocas_devolucoes': {
        's': ['', 'Olá', 'Por favor', 'Gostaria de solicitar', 'Quero abrir'],
        'a': [
            'preciso trocar o',
            'quero devolver a',
            'como solicito o estorno do',
            'desejo solicitar a troca da',
            'como funciona a devolucao do'
        ],
        'o': [
            'produto com defeito',
            'mesa que veio arranhada',
            'cadeira no prazo de 7 dias',
            'pedido cancelado',
            'item com avaria'
        ]
    },

    'reclamacoes': {
        's': ['', 'Urgente', 'Pessimo atendimento', 'Absurdo', 'Quero registrar'],
        'a': [
            'estou indignado com o',
            'quero fazer uma queixa do',
            'estou reclamando do',
            'produto veio quebrado e o',
            'atendimento horrivel do'
        ],
        'o': [
            'atraso na minha entrega',
            'servico de montagem',
            'sac que nao responde',
            'pos venda da loja',
            'estado do meu movel'
        ]
    },

    'logistica_entregas': {
        's': ['', 'Olá', 'Bom dia', 'Por gentileza', 'Preciso saber'],
        'a': [
            'onde esta o meu',
            'qual o prazo de entrega do',
            'como rastreio a',
            'qual a transportadora do',
            'quando chega o'
        ],
        'o': [
            'meu pedido',
            'codigo de rastreamento',
            'movel comprado',
            'status do envio',
            'agendamento da entrega'
        ]
    }
}


# Gerar 100 frases
amostras = []

random.seed(42)

for intencao, comp in templates.items():

    for _ in range(20):

        s = random.choice(comp['s'])
        a = random.choice(comp['a'])
        o = random.choice(comp['o'])

        frase = f"{s} {a} {o}".strip().capitalize()

        amostras.append({
            'texto': frase,
            'intencao': intencao
        })


# Criar DataFrame
df = pd.DataFrame(amostras)


# Salvar CSV
df.to_csv(
    'dataset_moveis_100.csv',
    index=False,
    encoding='utf-8'
)


print("Dataset de móveis criado com sucesso!")
print("\nDistribuição das intenções:")
print(df['intencao'].value_counts())


# ============================================================
# 2. DIVISÃO TREINO E TESTE
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    df['texto'],
    df['intencao'],
    test_size=0.30,
    random_state=42,
    stratify=df['intencao']
)


# ============================================================
# 3. PIPELINE TF-IDF + DECISION TREE
# ============================================================

pipeline_dt = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])


# ============================================================
# 4. TREINAR O MODELO
# ============================================================

pipeline_dt.fit(X_train, y_train)


# ============================================================
# 5. AVALIAÇÃO DO MODELO
# ============================================================

y_pred = pipeline_dt.predict(X_test)


print("\n========================================")
print("MATRIZ DE CONFUSÃO")
print("========================================")

print(confusion_matrix(y_test, y_pred))


print("\n========================================")
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("========================================")

print(classification_report(y_test, y_pred))


# ============================================================
# 6. PREPARAR TF-IDF PARA O FALLBACK
# ============================================================

vectorizer_fallback = TfidfVectorizer()

X_train_tfidf = vectorizer_fallback.fit_transform(X_train)


# ============================================================
# 7. CONFIGURAÇÃO DO FALLBACK
# ============================================================

LIMIAR_CONFIANCA = 0.50
LIMIAR_SIMILARIDADE = 0.20


# ============================================================
# 8. TESTES MANUAIS - 8 FRASES
# ============================================================

print("\n========================================")
print("TESTES MANUAIS - 8 FRASES")
print("========================================")


for i in range(1, 9):

    print(f"\n[Teste {i}/8]")

    frase = input("Digite a frase do cliente: ").strip()


    # Evita entrada vazia
    if frase == "":
        print(
            "Desculpe, não entendi sua solicitação. "
            "Encaminhando você para um atendente humano..."
        )
        continue


    # ========================================================
    # PREVISÃO DA INTENÇÃO
    # ========================================================

    intencao = pipeline_dt.predict([frase])[0]


    # ========================================================
    # PROBABILIDADE
    # ========================================================

    probs = pipeline_dt.predict_proba([frase])

    maior_prob = np.max(probs)


    # ========================================================
    # SIMILARIDADE COM O DATASET
    # ========================================================

    frase_tfidf = vectorizer_fallback.transform([frase])

    similaridades = cosine_similarity(
        frase_tfidf,
        X_train_tfidf
    )

    maior_similaridade = np.max(similaridades)


    # ========================================================
    # DECISÃO
    # ========================================================

    if (
        maior_prob >= LIMIAR_CONFIANCA
        and maior_similaridade >= LIMIAR_SIMILARIDADE
    ):

        print(f"Intenção identificada: {intencao}")
        print(f"Confiança: {maior_prob:.2%}")

    else:

        print(
            "Desculpe, não entendi sua solicitação. "
            "Encaminhando você para um atendente humano..."
        )


Dataset de móveis criado com sucesso!

Distribuição das intenções:
intencao
vendas                20
suporte               20
trocas_devolucoes     20
reclamacoes           20
logistica_entregas    20
Name: count, dtype: int64

MATRIZ DE CONFUSÃO
[[4 0 0 0 2]
 [1 4 1 0 0]
 [0 0 6 0 0]
 [0 0 1 5 0]
 [0 0 0 1 5]]

RELATÓRIO DE CLASSIFICAÇÃO
                    precision    recall  f1-score   support

logistica_entregas       0.80      0.67      0.73         6
       reclamacoes       1.00      0.67      0.80         6
           suporte       0.75      1.00      0.86         6
 trocas_devolucoes       0.83      0.83      0.83         6
            vendas       0.71      0.83      0.77         6

          accuracy                           0.80        30
         macro avg       0.82      0.80      0.80        30
      weighted avg       0.82      0.80      0.80        30


TESTES MANUAIS - 8 FRASES

[Teste 1/8]
Digite a frase do cliente: Quero comprar um sofá
Intenção identificada: troc